<a href="https://colab.research.google.com/github/LailaBulh/Procesamiento_Lenguaje_Natural/blob/main/Agrupamiento_textos_LB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Clasificación de texto**

**Nombre:** Laila Montserrat Bulhosen Ramos **Matrícula**: 263166

**Docente:** Dr. Rogelio Florencia Juarez

**Materia:** Procesamiento de Lenguaje Natural



**Link Github:** [Agrupamiento_texto_LB](https://github.com/LailaBulh/Procesamiento_Lenguaje_Natural/blob/main/Agrupamiento_textos_LB.ipynb)

**Fecha:** Mayor 2026


### **1. Carga de datos**

In [2]:
from sklearn.datasets import fetch_20newsgroups
dataset = fetch_20newsgroups(subset = 'all', remove = ('headers','footers','quotes'))
docs_raw = dataset.data

In [8]:
### Vista previa
print(docs_raw[0])



I am sure some bashers of Pens fans are pretty confused about the lack
of any kind of posts about the recent Pens massacre of the Devils. Actually,
I am  bit puzzled too and a bit relieved. However, I am going to put an end
to non-PIttsburghers' relief with a bit of praise for the Pens. Man, they
are killing those Devils worse than I thought. Jagr just showed you why
he is much better than his regular season stats. He is also a lot
fo fun to watch in the playoffs. Bowman should let JAgr have a lot of
fun in the next couple of games since the Pens are going to beat the pulp out of Jersey anyway. I was very disappointed not to see the Islanders lose the final
regular season game.          PENS RULE!!!




### **1.1 Preprocesamiento de datos**

Conversión a minúsculas

In [13]:
docs_min = [doc.lower() for doc in docs_raw]

### Verificación de los cambios a minúsculas
print(docs_min[0])



i am sure some bashers of pens fans are pretty confused about the lack
of any kind of posts about the recent pens massacre of the devils. actually,
i am  bit puzzled too and a bit relieved. however, i am going to put an end
to non-pittsburghers' relief with a bit of praise for the pens. man, they
are killing those devils worse than i thought. jagr just showed you why
he is much better than his regular season stats. he is also a lot
fo fun to watch in the playoffs. bowman should let jagr have a lot of
fun in the next couple of games since the pens are going to beat the pulp out of jersey anyway. i was very disappointed not to see the islanders lose the final
regular season game.          pens rule!!!




Eliminación de puntuaciones

In [14]:
import re

In [16]:
docs_clean = [re.sub(r'[^\w\s]', '', doc) for doc in docs_min]

### Verificación de remover puntuaciones
print(docs_clean[0])



i am sure some bashers of pens fans are pretty confused about the lack
of any kind of posts about the recent pens massacre of the devils actually
i am  bit puzzled too and a bit relieved however i am going to put an end
to nonpittsburghers relief with a bit of praise for the pens man they
are killing those devils worse than i thought jagr just showed you why
he is much better than his regular season stats he is also a lot
fo fun to watch in the playoffs bowman should let jagr have a lot of
fun in the next couple of games since the pens are going to beat the pulp out of jersey anyway i was very disappointed not to see the islanders lose the final
regular season game          pens rule




Eliminación de stopwords en inglés y tokenización

In [22]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [28]:
nltk.download('stopwords')
nltk.download('punkt')
english_stopwords = set(stopwords.words("english"))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [30]:
nltk.download('punkt_tab') # Download the punkt_tab tokenizer model


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [32]:
stopwords_eng = stopwords.words('english')

all_tokens = [word_tokenize(doc, language='english') for doc in docs_clean]

### Verificacion de tokens para el primer record
print(all_tokens[0][0:10])

['i', 'am', 'sure', 'some', 'bashers', 'of', 'pens', 'fans', 'are', 'pretty']


### **2 Entrenamiento de embeddings**

#### **2.1 Entrenamiento de embeddings con Word2Vec**

In [42]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 23.0 MB/s eta 0:00:00


In [43]:

from gensim.models import Word2Vec
from sklearn.cluster import KMeans

In [45]:
### Entrenar con vectores de tamaño 100

model = Word2Vec(
    sentences = all_tokens,
    vector_size = 100,
    window = 5,
    min_count = 3,
    workers = 4,
    sg = 1
)

model.train(all_tokens, total_examples=len(all_tokens), epochs=10)

(25532296, 33621110)

In [46]:

embeddings = model.wv.vectors
k = 6
kmeans = KMeans(n_clusters=k, random_state=42)
kmeans.fit(embeddings)

labels = kmeans.labels_

#### **2.2 Determinar valor de K**

##### Método del codo

In [48]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import numpy as np

In [49]:
wcss = []
K = range(2, 20)

# Function to get document vector by averaging word vectors
def get_document_vector(word_list, w2v_model):
    # Filter out words not in the model's vocabulary and get their vectors
    vectors = [w2v_model.wv[word] for word in word_list if word in w2v_model.wv]
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        # Return a zero vector if no words are found in vocabulary or list is empty
        return np.zeros(w2v_model.vector_size)

# Create document vectors by averaging word embeddings
# 'model' is the trained Word2Vec model from cell VcmYwkReTdsD
# 'all_tokens' is a list of lists of words (tokenized documents)
document_vectors = [get_document_vector(doc_tokens, model) for doc_tokens in all_tokens]

# Convert the list of document vectors to a NumPy array for KMeans
X_doc_vectors = np.array(document_vectors)

# Ensure there are enough samples to perform clustering
if X_doc_vectors.shape[0] == 0:
    print("No document vectors could be created. Cannot perform KMeans.")
else:
    for k in K:
      # Added random_state for reproducibility and n_init for best practice
      kmeans = KMeans(n_clusters = k, random_state=42, n_init=10)
      kmeans.fit(X_doc_vectors) # Fit KMeans on the document vectors
      wcss.append(kmeans.inertia_)

    plt.plot(K, wcss)

    plt.xlabel('Número de clusters(K)')
    plt.ylabel('Suma de cuadrados(WCSS)')
    plt.title('Método del codo') # Corrected typo from tittle to title

    plt.show()

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (18846,) + inhomogeneous part.

##### Coeficiente de Silhoutte

#### **2.3 Gráficas correspondientes**

#### **Entrenamiento K-Meand usando K óptimo**

#### **Cálculo y reporte de metricas**